# 2. Feature Engineering

Create time features, encoding features, and scaled numeric features for model training.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bsdf.config import get_config
from bsdf.data import load_processed_data, split_time_ordered_data
from bsdf.features import add_time_features, build_preprocessor, create_feature_target_split, FEATURE_COLUMNS

CONFIG = get_config(PROJECT_ROOT)

In [2]:
data = load_processed_data(CONFIG)
featured_data = add_time_features(data)

train_data, test_data = split_time_ordered_data(featured_data, CONFIG.test_fraction)
X_train, y_train = create_feature_target_split(train_data)
X_test, y_test = create_feature_target_split(test_data)

print(f"Train rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
X_train.head()

Train rows: 13,903
Test rows: 3,476


,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,hour_sin,hour_cos,month_sin,month_cos
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,0.000000,1.000000,0.5,0.866025
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,0.258819,0.965926,0.5,0.866025
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,0.500000,0.866025,0.5,0.866025
3,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,0.707107,0.707107,0.5,0.866025
4,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0.866025,0.500000,0.5,0.866025


## Time Features

Hour and month are represented both as categorical variables and as cyclical sine/cosine variables so the model understands that hour 23 is close to hour 0.

In [3]:
featured_data[["timestamp", "hr", "hour_sin", "hour_cos", "mnth", "month_sin", "month_cos"]].head(10)

,timestamp,hr,hour_sin,hour_cos,mnth,month_sin,month_cos
0,2011-01-01 00:00:00,0,0.000000,1.000000e+00,1,0.5,0.866025
1,2011-01-01 01:00:00,1,0.258819,9.659258e-01,1,0.5,0.866025
2,2011-01-01 02:00:00,2,0.500000,8.660254e-01,1,0.5,0.866025
3,2011-01-01 03:00:00,3,0.707107,7.071068e-01,1,0.5,0.866025
4,2011-01-01 04:00:00,4,0.866025,5.000000e-01,1,0.5,0.866025
5,2011-01-01 05:00:00,5,0.965926,2.588190e-01,1,0.5,0.866025
6,2011-01-01 06:00:00,6,1.000000,6.123234e-17,1,0.5,0.866025
7,2011-01-01 07:00:00,7,0.965926,-2.588190e-01,1,0.5,0.866025
8,2011-01-01 08:00:00,8,0.866025,-5.000000e-01,1,0.5,0.866025
9,2011-01-01 09:00:00,9,0.707107,-7.071068e-01,1,0.5,0.866025


## Encoding And Scaling

Categorical features are one-hot encoded. Numeric weather and cyclical features are scaled for linear regression and passed through unchanged for tree-based models.

In [4]:
scaled_preprocessor = build_preprocessor(scale_numeric=True)
encoded_scaled_sample = scaled_preprocessor.fit_transform(X_train.head(20))
print(f"Original feature count: {len(FEATURE_COLUMNS)}")
print(f"Encoded/scaled feature count: {encoded_scaled_sample.shape[1]}")
pd.DataFrame(encoded_scaled_sample).head()

Original feature count: 16
Encoded/scaled feature count: 37


,0,1,2,3,4,5,6,7,8,9,...,27,28,29,30,31,32,33,34,35,36
0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.929268,-0.858588,0.454061,-1.051497,-0.163581,1.707281,5.551115e-17,0.0
1,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-1.135772,-1.061361,0.237841,-1.051497,0.199531,1.656781,5.551115e-17,0.0
2,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-1.135772,-1.061361,0.237841,-1.051497,0.537897,1.508721,5.551115e-17,0.0
3,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,-0.929268,-0.858588,-0.843256,-1.051497,0.828459,1.273192,5.551115e-17,0.0
4,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,-0.929268,-0.858588,-0.843256,-1.051497,1.051415,0.966244,5.551115e-17,0.0


In [5]:
featured_data.to_csv(CONFIG.processed_dir / "hourly_featured.csv", index=False)
train_data.to_csv(CONFIG.processed_dir / "train_featured.csv", index=False)
test_data.to_csv(CONFIG.processed_dir / "test_featured.csv", index=False)
print("Feature datasets saved to data/processed.")

Feature datasets saved to data/processed.
